# 03 CoM and Geometry Normalization

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`exp_cmb004_cross_diagnostic.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb004_cross_diagnostic.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb004_geometry_normalization.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb004_geometry_normalization.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`pixel_transform.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/pixel_transform.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)
- [`exp_cmb004_compare.py`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb004_compare.py) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'05_model_optimize/MLP/Combined/scripts/exp_cmb004_cross_diagnostic.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '4cf7675dfa63303e8124ae30aeacedb1f97b4cfd'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb004_geometry_normalization.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': '8bba16ba48f88be3a2ed31138be75a82824236db'}, '05_model_optimize/MLP/Combined/scripts/pixel_transform.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'a56a88b602a98d0cf26dfa6ee36c31e2b9d08a7a'}, '05_model_optimize/MLP/Combined/scripts/exp_cmb004_compare.py': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'd5c097d4ca79560ad8a512c2aaa88e95b5bce791'}}

## exp_cmb004_cross_diagnostic

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb004_cross_diagnostic.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb004_cross_diagnostic.py)  
Git blob: `4cf7675dfa63303e8124ae30aeacedb1f97b4cfd`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-004 post hoc 4-way input/weights control for explanation.

For each writer-held-out fold and normalized method:
  A = original held-out image + original frozen Combined Baseline MLP
  B = transformed held-out image + SAME original frozen MLP
  C = SAME transformed held-out image + new MLP trained with normalization
  D = original held-out image + new normalized MLP (OOD diagnostic only)
The critical two comparisons are A->B (pure input intervention) and B->C
(same input, different traini



In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb004_cross_diagnostic.py
#!/usr/bin/env python3
"""EXP-CMB-004 post hoc 4-way input/weights control for explanation.

For each writer-held-out fold and normalized method:
  A = original held-out image + original frozen Combined Baseline MLP
  B = transformed held-out image + SAME original frozen MLP
  C = SAME transformed held-out image + new MLP trained with normalization
  D = original held-out image + new normalized MLP (OOD diagnostic only)
The critical two comparisons are A->B (pure input intervention) and B->C
(same input, different training). No new training, no Test, no tuning.
"""
from __future__ import annotations
import csv
import hashlib
import io
import json
import sys
import zipfile
from collections import defaultdict
from pathlib import Path
import joblib
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

ROOT=Path(__file__).resolve().parents[4]
CMB=ROOT/"05_model_optimize"/"MLP"/"Combined"
sys.path.insert(0,str(CMB/"scripts"))
import exp_cmb002_evidence as core
import exp_cmb004_geometry_normalization as stage

OUT=CMB/"results"/"Geometry_Normalization"
METHODS=stage.METHODS
LABELS=stage.LABELS
WRITERS=stage.WRITERS
RIVAL={"0":"D","D":"0","8":"B","B":"8"}

def read(path):
    with path.open("r",encoding="utf-8-sig",newline="") as f:
        return list(csv.DictReader(f))

def output(path,rows):
    rows=list(rows)
    assert rows
    with path.open("w",encoding="utf-8-sig",newline="") as f:
        wr=csv.DictWriter(f,fieldnames=list(rows[0]))
        wr.writeheader()
        wr.writerows(rows)

def load_new_models(method):
    manifest=json.loads((OUT/method/"run_manifest.json").read_text(encoding="utf-8"))
    result={}
    with zipfile.ZipFile(OUT/method/"trained_models.zip") as z:
        for w in WRITERS:
            fname=f"{method}_heldout_{w}_mlp.joblib"
            data=z.read(fname)
            assert hashlib.sha256(data).hexdigest()==manifest["trained_models_sha256"][fname]
            result[w]=joblib.load(io.BytesIO(data))
    return result

def score(y,p,classes=None):
    classes=np.arange(16) if classes is None else classes
    pr,re,f1,s=precision_recall_fscore_support(y,p,labels=classes,zero_division=0)
    return {"n":len(y),"correct":int(np.sum(y==p)),
            "accuracy":float(np.mean(y==p)),
            "macro_f1":float(np.mean(f1)),
            "macro_precision":float(np.mean(pr)),
            "macro_recall":float(np.mean(re))}

def main():
    manifest,pred,table,hand,labels=core.load_inputs()
    frozen=core.load_models(manifest)
    goal=json.loads((OUT/"AGGREGATE_MANIFEST.json").read_text(encoding="utf-8"))["size_target"]
    assert goal==20
    all_rows=[];focus=[];pergroup=[];bywriter=[];pair_counts=[]
    summary=[]
    for method in METHODS:
        trained=load_new_models(method)
        target=read(OUT/method/"paired_predictions.csv")
        assert len(target)==2560
        method_p={}
        idx=0
        for writer in WRITERS:
            raw=np.asarray(hand[writer],dtype=np.float32).reshape(640,784)
            y=np.asarray(labels[writer],dtype=int)
            transform,scales,_=stage.apply_method(raw,method,goal)
            old=frozen[f"combined_heldout_{writer}_mlp.joblib"]
            new=trained[writer]
            a=old.predict(raw).astype(int)
            b=old.predict(transform).astype(int)
            c=new.predict(transform).astype(int)
            d=new.predict(raw).astype(int)
            for i in range(640):
                r=table[writer,i]
                assert LABELS[a[i]]==r["combined_pred"],(writer,i,"baseline")
                q=target[idx];idx+=1
                assert (q["writer"],q["index_in_writer"],q["actual"],q["prediction"])==(
                    str(writer),str(i),LABELS[y[i]],LABELS[c[i]]),(method,writer,i)
            method_p[writer]={"A":a,"B":b,"C":c,"D":d,"y":y}
            bywriter.extend({"method":method,"writer":writer,"variant":variant,**score(y,p)}
                for variant,p in (("A_baseline_raw",a),("B_baseline_transformed",b),
                                  ("C_retrained_transformed",c),("D_retrained_raw",d)))
            for i in range(640):
                t=LABELS[y[i]]
                all_rows.append({"method":method,"writer":writer,"index_in_writer":i,
                 "true_label":t,"A":LABELS[a[i]],"B":LABELS[b[i]],
                 "C":LABELS[c[i]],"D":LABELS[d[i]],
                 "A_correct":int(a[i]==y[i]),"B_correct":int(b[i]==y[i]),
                 "C_correct":int(c[i]==y[i]),"D_correct":int(d[i]==y[i])})
            for actual,rival in stage.PAIRS:
                subset=y==LABELS.index(actual)
                for variant,p in (("A_baseline_raw",a),("B_baseline_transformed",b),
                                  ("C_retrained_transformed",c),("D_retrained_raw",d)):
                    pair_counts.append({"method":method,"writer":writer,
                     "actual":actual,"rival":rival,"variant":variant,
                     "errors":int(np.sum(p[subset]==LABELS.index(rival)))})
            # True-vs-fixed-rival logits for 4-WAY on every 0/D/8/B image.
            outs={"A":core.logits(old,raw),"B":core.logits(old,transform),
                  "C":core.logits(new,transform),"D":core.logits(new,raw)}
            for i in range(640):
                actual=LABELS[y[i]]
                if actual not in RIVAL:continue
                other=RIVAL[actual]
                a_i,b_i,c_i,d_i=(float(outs[v][i,y[i]]-
                            outs[v][i,LABELS.index(other)]) for v in ("A","B","C","D"))
                focus.append({"method":method,"writer":writer,"index_in_writer":i,
                    "actual":actual,"rival":other,
                    "baseline_correct":int(a[i]==y[i]),
                    "input_only_correct":int(b[i]==y[i]),
                    "retrained_correct":int(c[i]==y[i]),
                    "original_A_margin":a_i,
                    "frozen_B_margin":b_i,
                    "retrained_C_margin":c_i,
                    "OOD_retrained_D_margin":d_i,
                    "input_only_margin_delta_B_minus_A":b_i-a_i,
                    "weights_on_identical_input_margin_delta_C_minus_B":c_i-b_i,
                    "size_ratio":float(scales[i])})
        assert idx==2560
        for variant in ("A","B","C","D"):
            y=np.concatenate([method_p[w]["y"] for w in WRITERS])
            p=np.concatenate([method_p[w][variant] for w in WRITERS])
            s=score(y,p)
            summary.append({"method":method,"variant":{
              "A":"A_baseline_raw","B":"B_baseline_transformed",
              "C":"C_retrained_transformed","D":"D_retrained_raw"}[variant],**s})
            for group,mask in (("digits",y<10),("letters",y>=10)):
                sc=score(y[mask],p[mask],
                    np.arange(10) if group=="digits" else np.arange(10,16))
                pergroup.append({"method":method,"variant":variant,"group":group,**sc})

    output(OUT/"cross_4way_summary.csv",summary)
    output(OUT/"cross_4way_writers.csv",bywriter)
    output(OUT/"cross_4way_digit_letter.csv",pergroup)
    output(OUT/"cross_4way_pair_errors.csv",pair_counts)
    output(OUT/"cross_4way_paired_predictions.csv",all_rows)
    output(OUT/"cross_4way_focus_margins.csv",focus)

    lines=["# EXP-CMB-004 — 학습 효과/입력 변형 효과 분리 대조",
        "",
        "**A** 기존 입력+기존 MLP / **B** 전처리 입력+동일한 기존 MLP",
        "**C** B와 정확히 같은 전처리 입력+해당 방법으로 새로 학습한 MLP / **D** 원본 입력+새 MLP(훈련 분포 밖 대조)",
        "",
        "A→B는 가중치를 고정한 입력 조작의 효과, B→C는 변형 입력을 고정한 재학습 효과를 분리하여 비교한다.",
        "D는 배포용 Pipeline이 아니며 OOD diagnostics로만 읽어야 한다.",
        "",
        "| Method | A old/raw | B old/transformed | C new/transformed | D new/raw (OOD) | Input only B−A | Model adaptation C−B |",
        "|---|---:|---:|---:|---:|---:|---:|"]
    for method in METHODS:
        ss=[s for s in summary if s["method"]==method]
        a,b,c,d=[100*s["accuracy"] for s in ss]
        lines.append("| {} | {:.3f}% | {:.3f}% | {:.3f}% | {:.3f}% | {:+.3f}pp | {:+.3f}pp |".format(
            method,a,b,c,d,b-a,c-b))
    lines+=["",
        "### 그룹별 대비와 부작용",
        "",
        "| Method | Group | A old/raw | B old/trans | C new/trans | B−A | C−B |",
        "|---|---|---:|---:|---:|---:|---:|"]
    for method in METHODS:
        for group in ("digits","letters"):
            ss=[s for s in pergroup if s["method"]==method and s["group"]==group]
            a,b,c=[100*s["accuracy"] for s in ss[:3]]
            lines.append("| {} | {} | {:.3f}% | {:.3f}% | {:.3f}% | {:+.3f}pp | {:+.3f}pp |".format(
                method,group,a,b,c,b-a,c-b))
    lines+=["",
        "**Interpretation constraint:** Even B→C compares independently refitted models, not a direct causal identification of one particular weight. OOF authors were repeatedly examined in earlier studies; extrapolations require new writers. Final project Test was not used.",
        ""]
    (OUT/"CROSS_4WAY_SUMMARY.md").write_text("\n".join(lines),encoding="utf-8")
    print((OUT/"CROSS_4WAY_SUMMARY.md").read_text(encoding="utf-8"),flush=True)

if __name__=="__main__":
    main()


## exp_cmb004_geometry_normalization

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb004_geometry_normalization.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb004_geometry_normalization.py)  
Git blob: `8bba16ba48f88be3a2ed31138be75a82824236db`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

EXP-CMB-004: preregistered explanatory BBox-Size vs CoM-Center study.

This is a new 4-Fold (Leave-One-Writer-Out) training comparison. No model/seed/
training-size tuning, and no accuracy threshold determines whether the study
answers its research question. The historical EXP-CMB-001 is a frozen baseline.
See logs/가설B/EXP-CMB-004_01_Plan_Geometry_Normalization.md for pre-run criteria.

All images are 28x28 grayscale 0..255. Only original official X_train keys
are read. NEVER access the original

CLI 인자 (정적 추출): `--out`, `--method`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb004_geometry_normalization.py
#!/usr/bin/env python3
"""EXP-CMB-004: preregistered explanatory BBox-Size vs CoM-Center study.

This is a new 4-Fold (Leave-One-Writer-Out) training comparison. No model/seed/
training-size tuning, and no accuracy threshold determines whether the study
answers its research question. The historical EXP-CMB-001 is a frozen baseline.
See logs/가설B/EXP-CMB-004_01_Plan_Geometry_Normalization.md for pre-run criteria.

All images are 28x28 grayscale 0..255. Only original official X_train keys
are read. NEVER access the original Test or professor Test datasets.
"""
from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import os
import sys
import time
import warnings
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import scipy
from scipy.ndimage import affine_transform
import sklearn
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from joblib import dump
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline

ROOT = Path(__file__).resolve().parents[4]
CMB = ROOT / "05_model_optimize" / "MLP" / "Combined"
sys.path.insert(0,str(CMB/"scripts"))
sys.path.insert(0,str(ROOT/"05_model_optimize"/"MLP"/"Normal"/"scripts"))
import exp_cmb002_evidence as evidence
from combined_mlp_writer_cv import load_official, load_writers, new_mlp
from Automatic_Centering_Transformer import center_batch, center_shifts, raw_centers
from pixel_transform import PixelScaler255

LABELS="0123456789ABCDEF"
WRITERS=(41,42,43,44)
METHODS=("bbox_center","bbox_size","com_center","size_then_com")
THRESHOLD=32
BASE=CMB/"results"/"EXP_CMB_001"
PAIRS=(("0","D"),("D","0"),("8","B"),("B","8"))

def csv_save(path,rows,headers=None):
    rows=list(rows)
    assert rows or headers, path
    path.parent.mkdir(parents=True,exist_ok=True)
    with path.open("w",newline="",encoding="utf-8-sig") as f:
        w=csv.DictWriter(f,fieldnames=headers or list(rows[0]))
        w.writeheader()
        w.writerows(rows)

def image_geometry(batch):
    a=np.asarray(batch,dtype=np.float32).reshape(-1,28,28)
    fg=a>=THRESHOLD
    assert np.all(fg.any(axis=(1,2))),"empty binary foreground"
    xx=fg.any(1)
    yy=fg.any(2)
    xmin=xx.argmax(1)
    xmax=27-xx[:,::-1].argmax(1)
    ymin=yy.argmax(1)
    ymax=27-yy[:,::-1].argmax(1)
    width=xmax-xmin+1
    height=ymax-ymin+1
    bbox_x=(xmax+xmin)/2.0
    bbox_y=(ymax+ymin)/2.0
    m=np.where(fg,a,0.0).astype(np.float64,copy=False)
    mass=m.sum((1,2))
    cx=(m*np.arange(28,dtype=float)[None,None,:]).sum((1,2))/mass
    cy=(m*np.arange(28,dtype=float)[None,:,None]).sum((1,2))/mass
    return dict(width=width,height=height,long_side=np.maximum(width,height),
                bbox_cx=bbox_x,bbox_cy=bbox_y,cx=cx,cy=cy,
                com_offset=np.hypot(cx-13.5,cy-13.5),
                bbox_offset=np.hypot(bbox_x-13.5,bbox_y-13.5),
                ink_ratio=fg.mean((1,2)),
                mass=a.sum((1,2),dtype=np.float64),
                edge_mass=np.sum(a[:,[0,1,26,27],:],axis=(1,2))+
                          np.sum(a[:,:, [0,1,26,27]],axis=(1,2)))

def apply_size(batch,target):
    a=np.asarray(batch,dtype=np.float32).reshape(-1,28,28)
    geo=image_geometry(a)
    factors=float(target)/geo["long_side"].astype(float)
    result=np.empty_like(a)
    for i in range(len(a)):
        inv=1.0/factors[i]
        center=(geo["bbox_cy"][i],geo["bbox_cx"][i])
        output=affine_transform(a[i],np.eye(2)*inv,
                 offset=(center[0]*(1-inv),center[1]*(1-inv)),
                 output_shape=(28,28),order=1,
                 mode="constant",cval=0.0,prefilter=False)
        result[i]=np.clip(output,0,255)
    return result.reshape(-1,784),factors

def apply_method(x,method,target):
    start=time.perf_counter()
    if method=="bbox_center":
        v=center_batch(x,"bbox")
        scales=np.ones(len(x),dtype=float)
    elif method=="bbox_size":
        v,scales=apply_size(x,target)
    elif method=="com_center":
        v=center_batch(x,"mass")
        scales=np.ones(len(x),dtype=float)
    elif method=="size_then_com":
        v,scales=apply_size(x,target)
        v=center_batch(v,"mass")
    elif method=="original":
        v=np.asarray(x,dtype=np.float32).reshape(-1,784)
        scales=np.ones(len(x),dtype=float)
    else:
        raise ValueError(method)
    assert np.isfinite(v).all() and v.min()>=0 and v.max()<=255
    return v.astype(np.float32,copy=False),scales,time.perf_counter()-start

def metrics(y,p,labels=np.arange(16)):
    pr,re,f1,s=precision_recall_fscore_support(y,p,labels=labels,zero_division=0)
    return dict(n=len(y),correct=int(np.sum(y==p)),accuracy=float(np.mean(y==p)),
                macro_precision=float(pr.mean()),macro_recall=float(re.mean()),
                macro_f1=float(f1.mean()))

def transitions(y,p0,p1):
    a=p0==y
    b=p1==y
    return dict(corrected=int(np.sum(~a&b)),
                induced=int(np.sum(a&~b)),
                both_wrong_same=int(np.sum(~a&~b&(p0==p1))),
                both_wrong_changed=int(np.sum(~a&~b&(p0!=p1))),
                both_correct=int(np.sum(a&b)),
                prediction_flip=int(np.sum(p0!=p1)),
                delta_correct=int(b.sum()-a.sum()))

def cmplot(out,label,y,p):
    cm=confusion_matrix(y,p,labels=np.arange(16))
    for kind in ("count","normalized"):
        dat=cm if kind=="count" else 100.0*cm/np.maximum(cm.sum(1,keepdims=True),1)
        fig,ax=plt.subplots(figsize=(9,7.8))
        limit=160 if kind=="count" else 100
        im=ax.imshow(dat,cmap="viridis",vmin=0,vmax=limit,interpolation="nearest")
        fig.colorbar(im,ax=ax,pad=.025,fraction=.04)
        ax.set(xticks=np.arange(16),yticks=np.arange(16),
           xticklabels=list(LABELS),yticklabels=list(LABELS),
           xlabel="Predicted",ylabel="Actual",title=label+" "+kind)
        for i in range(16):
            for j in range(16):
                if cm[i,j]:
                    ax.text(j,i,("%.1f"%dat[i,j] if kind=="normalized" else str(cm[i,j])),
                            ha="center",va="center",fontsize=5,
                            color="black" if dat[i,j]>.55*limit else "white")
        fig.tight_layout()
        fig.savefig(out/f"{label}_{kind}.png",dpi=150)
        plt.close(fig)
    return cm

def geo_summary(name,before,after,scales):
    a,b=image_geometry(before),image_geometry(after)
    startmass=np.maximum(a["mass"],1)
    return dict(sample=name,n=len(before),
      mean_long_before=float(a["long_side"].mean()),mean_long_after=float(b["long_side"].mean()),
      std_long_before=float(a["long_side"].std()),std_long_after=float(b["long_side"].std()),
      mean_com_offset_before=float(a["com_offset"].mean()),
      mean_com_offset_after=float(b["com_offset"].mean()),
      mean_bbox_offset_before=float(a["bbox_offset"].mean()),
      mean_bbox_offset_after=float(b["bbox_offset"].mean()),
      mean_abs_size_ratio_change=float(np.mean(np.abs(scales-1.0))),
      fraction_size_ratio_gt_1_10=float(np.mean(np.abs(scales-1)>=.10)),
      fraction_changed=float(np.mean(np.any(np.abs(np.asarray(before).reshape(-1,784)-after)>.1,axis=1))),
      mean_abs_ink_mass_fraction_change=float(np.mean(np.abs((b["mass"]-a["mass"])/startmass))),
      mean_positive_edge_mass_fraction=float(np.mean(b["edge_mass"]/np.maximum(b["mass"],1))),
      fraction_abs_com_change_over_half_px=float(np.mean(
        np.hypot(b["cx"]-a["cx"],b["cy"]-a["cy"])>.5)))

def find_unexpected_pixel_changes(before,after,name):
    if name=="bbox_size" or name=="size_then_com":
        return
    a=np.asarray(before).reshape(-1,28,28)
    b=np.asarray(after).reshape(-1,28,28)
    if name in ("bbox_center","com_center"):
        # no interpolation in integer center-only methods
        assert np.all(np.isin(np.unique(b),np.unique(a)))
    assert np.isfinite(b).all()

def per_image_explanation(method,held,old_y,old_p,new_p,original,changed,ratio):
    g0=image_geometry(original)
    g1=image_geometry(changed)
    # For all 640 in this writer, score the 4 confusion directions using the
    # competing class fixed by the *true* label, for posthoc research ONLY.
    rows=[]
    for i in range(len(old_y)):
        t=LABELS[int(old_y[i])]
        if t not in ("0","D","8","B"):
            continue
        rival={"0":"D","D":"0","8":"B","B":"8"}[t]
        rows.append({"method":method,"writer":held,"index_in_writer":i,
          "actual":t,"competitor":rival,
          "baseline_pred":LABELS[int(old_p[i])],"new_pred":LABELS[int(new_p[i])],
          "baseline_correct":int(old_p[i]==old_y[i]),
          "new_correct":int(new_p[i]==old_y[i]),
          "transition":"corrected" if old_p[i]!=old_y[i] and new_p[i]==old_y[i] else
                "new_error" if old_p[i]==old_y[i] and new_p[i]!=old_y[i] else
                "persistent_error" if new_p[i]!=old_y[i] else "both_correct",
          "scale_factor":float(ratio[i]),
          "bbox_long_before":int(g0["long_side"][i]),
          "bbox_long_after":int(g1["long_side"][i]),
          "bbox_change":int(g1["long_side"][i]-g0["long_side"][i]),
          "com_x_before":float(g0["cx"][i]),"com_y_before":float(g0["cy"][i]),
          "com_x_after":float(g1["cx"][i]),"com_y_after":float(g1["cy"][i]),
          "com_shift_px":float(np.hypot(g1["cx"][i]-g0["cx"][i],
                                      g1["cy"][i]-g0["cy"][i])),
          "mass_change_ratio":float((g1["mass"][i]-g0["mass"][i])/max(g0["mass"][i],1))})
    return rows

def plot_examples(root,original,transformed,method,held):
    if held!=41 and held!=43:return
    ids=[32,104,11,317]
    fig,axes=plt.subplots(len(ids),2,figsize=(6,11))
    for j,i in enumerate(ids):
        axes[j,0].imshow(original[i].reshape(28,28),cmap="gray",vmin=0,vmax=255)
        axes[j,1].imshow(transformed[i].reshape(28,28),cmap="gray",vmin=0,vmax=255)
        for k in (0,1):axes[j,k].axis("off")
        axes[j,0].set_title(f"Writer {held} #{i} raw")
        axes[j,1].set_title(f"{method}")
    fig.tight_layout()
    fig.savefig(root/f"examples_{method}_writer_{held}.png",dpi=150)
    plt.close(fig)

def main():
    p=argparse.ArgumentParser()
    p.add_argument("--out",type=Path,required=True)
    p.add_argument("--method",choices=METHODS,required=True)
    args=p.parse_args()
    out=args.out.resolve()
    if (out/"metrics.csv").exists():
        raise RuntimeError("Existing result, do not overwrite: "+str(out))
    out.mkdir(parents=True,exist_ok=True)
    (out/"figures").mkdir(exist_ok=True)
    start=time.perf_counter()
    old_manifest,orig_pred,oldtab,raw_oof,old_labels=evidence.load_inputs()
    official=load_official(ROOT/old_manifest["input_data_paths"]["official"])
    writerdata=load_writers(ROOT/old_manifest["input_data_paths"]["handmade_directory"])
    assert np.array_equal(official["train_idx"],
       np.load(BASE/"official_split_indices.npz")["train_idx"])
    assert np.array_equal(official["val_idx"],
       np.load(BASE/"official_split_indices.npz")["val_idx"])
    goal=int(np.median(image_geometry(official["X_train"])["long_side"]))
    assert 10<=goal<=26
    print("SOURCE CHECKS PASSED / official-only BBox long-side median target",goal,flush=True)
    t0=time.perf_counter()
    off_train,rs_train,_=apply_method(official["X_train"],args.method,goal)
    off_val,rs_val,_=apply_method(official["X_val"],args.method,goal)
    authors={}
    author_ratios={}
    for w in WRITERS:
        authors[w],author_ratios[w],_=apply_method(writerdata[w]["X"],args.method,goal)
        find_unexpected_pixel_changes(writerdata[w]["X"],authors[w],args.method)
    preprocessing_seconds=time.perf_counter()-t0
    diag=[{"method":args.method,**geo_summary("official_train",official["X_train"],off_train,rs_train)},
          {"method":args.method,**geo_summary("official_val",official["X_val"],off_val,rs_val)}]
    for w in WRITERS:
        diag.append({"method":args.method,**geo_summary("writer_"+str(w),writerdata[w]["X"],authors[w],author_ratios[w])})
        plot_examples(out/"figures",writerdata[w]["X"],authors[w],args.method,w)
    csv_save(out/"geometry_diagnostics.csv",diag)

    # Historical OOF is the reference and MUST never be re-fitted or changed.
    base_correct={w:np.array([LABELS.index(oldtab[w,i]["combined_pred"]) for i in range(640)]) for w in WRITERS}
    yref={w:writerdata[w]["y"] for w in WRITERS}
    for w in WRITERS:
        assert np.array_equal(yref[w],
            np.array([LABELS.index(oldtab[w,i]["true_label"]) for i in range(640)]))
    old_models=evidence.load_models(old_manifest)
    preds={}
    official_rows=[];writer_rows=[];class_rows=[];paired=[];explanation=[]
    fit_rows=[];loss_rows=[];geoms=[];costs=[]
    model_data=[]
    for held in WRITERS:
        startfold=time.perf_counter()
        tr_authors=tuple(w for w in WRITERS if w!=held)
        X=np.concatenate([off_train]+[authors[w] for w in tr_authors])
        y=np.concatenate([official["y_train"]]+[yref[w] for w in tr_authors])
        assert X.shape==(78720,784) and y.shape==(78720,)
        m=new_mlp()
        tfit=time.perf_counter()
        with warnings.catch_warnings(record=True) as warns:
            warnings.simplefilter("always")
            m.fit(X,y)
        fitsec=time.perf_counter()-tfit
        net=m.named_steps["model"]
        assert m.named_steps["scale_0_1"].__class__ is PixelScaler255
        assert net.get_params()["random_state"]==42
        assert net.n_features_in_==784
        tpred=time.perf_counter()
        pred=m.predict(authors[held]).astype(int)
        pred_seconds=time.perf_counter()-tpred
        preds[held]=pred
        tdev=time.perf_counter()
        official_pred=m.predict(off_val).astype(int)
        dev_seconds=time.perf_counter()-tdev
        v=metrics(yref[held],pred)
        ov=metrics(official["y_val"],official_pred)
        orig=base_correct[held]
        tr=transitions(yref[held],orig,pred)
        writer_rows.append({"method":args.method,"writer":held,**v,**tr,
           "baseline_accuracy":float(np.mean(orig==yref[held])),
           "delta_accuracy_pp":float(100*(v["accuracy"]-np.mean(orig==yref[held])))})
        official_rows.append({"method":args.method,"writer":held,**ov,
           "fit_seconds":fitsec,"predict_seconds":dev_seconds})
        fit_rows.append({"method":args.method,"heldout":held,"train_n":len(X),
            "train_authors":",".join(map(str,tr_authors)),
            "fit_seconds":fitsec,"n_iter":int(net.n_iter_),"train_loss":float(net.loss_),
            "prediction_seconds_640":pred_seconds,
            "official_prediction_seconds_19200":dev_seconds,
            "warnings":" | ".join(str(x.message) for x in warns),
            "fold_wall_seconds":time.perf_counter()-startfold})
        loss_rows.extend({"method":args.method,"writer":held,
            "iteration":i+1,"training_loss":float(loss)} for i,loss in enumerate(net.loss_curve_))
        for j,cl in enumerate(LABELS):
            z=yref[held]==j
            origok=int(np.sum(orig[z]==j))
            newok=int(np.sum(pred[z]==j))
            class_rows.append({"method":args.method,"writer":held,"class":cl,"n":int(z.sum()),
                "old_correct":origok,"new_correct":newok,
                "corrected":int(np.sum((orig[z]!=j)&(pred[z]==j))),
                "induced":int(np.sum((orig[z]==j)&(pred[z]!=j)))})
        # Per image metric includes raw bbox/CoM change and label-aware
        # diagnostic rivals only; NEVER used at transform fitting/inference.
        explanation.extend(per_image_explanation(args.method,held,yref[held],
            orig,pred,writerdata[held]["X"],authors[held],author_ratios[held]))
        for i in range(640):
            paired.append({"method":args.method,"writer":held,"index_in_writer":i,
                "actual":LABELS[int(yref[held][i])],
                "baseline_pred":LABELS[int(orig[i])],
                "prediction":LABELS[int(pred[i])],
                "baseline_correct":int(orig[i]==yref[held][i]),
                "correct":int(pred[i]==yref[held][i])})
        # record margins for focus 0/D/8/B, not class-conditioned decisions
        g=evidence.logits(m,authors[held])
        baseline_logits=evidence.logits(old_models["combined_heldout_"+str(held)+"_mlp.joblib"], writerdata[held]["X"])
        for j,cl in enumerate(yref[held]):
            actual=LABELS[int(cl)]
            if actual in ("0","D","8","B"):
                rival={"0":"D","D":"0","8":"B","B":"8"}[actual]
                index=next(i for i in range(len(explanation)-160,len(explanation))
                           if explanation[i]["writer"]==held and
                              explanation[i]["index_in_writer"]==j)
                explanation[index]["actual_vs_rival_logit"]=float(g[j,int(cl)]-g[j,LABELS.index(rival)])
                explanation[index]["baseline_margin"]=float(baseline_logits[j,int(cl)]-baseline_logits[j,LABELS.index(rival)])
                explanation[index]["margin_change"]=explanation[index]["actual_vs_rival_logit"]-explanation[index]["baseline_margin"]
                explanation[index]["predicted_rival"]=int(pred[j]==LABELS.index(rival))
        mem=io.BytesIO()
        dump(m,mem,compress=3)
        payload=mem.getvalue()
        model_data.append((f"{args.method}_heldout_{held}_mlp.joblib",payload))
        cmplot(out/"figures",args.method+"_writer_"+str(held),yref[held],pred)
        print(args.method,held,"OOF",v["accuracy"],"Val",ov["accuracy"],
              "fit",round(fitsec,1),flush=True)
        del m,X,y
    whole_y=np.concatenate([yref[w] for w in WRITERS])
    whole_p=np.concatenate([preds[w] for w in WRITERS])
    whole_base=np.concatenate([base_correct[w] for w in WRITERS])
    v=metrics(whole_y,whole_p)
    tr=transitions(whole_y,whole_base,whole_p)
    assert len(whole_y)==2560 and np.sum(whole_base==whole_y)==2080
    overall=[{"method":args.method,**v,**tr,"baseline_accuracy":.8125,
       "delta_accuracy_pp":100*(v["accuracy"]-.8125),
       "preprocessing_seconds":preprocessing_seconds,
       "wall_seconds":time.perf_counter()-start}]
    groups=[]
    for label,z in (("digits",whole_y<10),("letters",whole_y>=10)):
        y=whole_y[z];pa=whole_p[z];ba=whole_base[z]
        labels=np.arange(10) if label=="digits" else np.arange(10,16)
        groups.append({"method":args.method,"group":label,**metrics(y,pa,labels),
                      **transitions(y,ba,pa),"baseline_accuracy":float(np.mean(y==ba))})
    conf=[]
    for actual,rival in PAIRS:
        cl=LABELS.index(actual);cv=LABELS.index(rival)
        for w in WRITERS:
            z=yref[w]==cl
            conf.append({"method":args.method,"writer":w,"actual":actual,"rival":rival,
              "baseline_errors":int(np.sum(base_correct[w][z]==cv)),
              "new_errors":int(np.sum(preds[w][z]==cv))})
    cmplot(out/"figures",args.method+"_pooled",whole_y,whole_p)
    csv_save(out/"metrics.csv",overall)
    csv_save(out/"writer_metrics.csv",writer_rows)
    csv_save(out/"official_val_metrics.csv",official_rows)
    csv_save(out/"class_metrics.csv",class_rows)
    csv_save(out/"group_metrics.csv",groups)
    csv_save(out/"pair_errors.csv",conf)
    csv_save(out/"fit_metrics.csv",fit_rows)
    csv_save(out/"loss_curves.csv",loss_rows)
    csv_save(out/"paired_predictions.csv",paired)
    csv_save(out/"focus_mechanism.csv",explanation)
    fig,ax=plt.subplots(figsize=(9,5))
    for w in WRITERS:
        c=[r for r in loss_rows if r["writer"]==w]
        ax.plot([r["iteration"] for r in c],[r["training_loss"] for r in c],label="heldout "+str(w))
    ax.set(xlabel="MLP iteration",ylabel="Training Loss",title=args.method+" | identical MLP settings")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out/"figures"/"training_loss.png",dpi=170)
    plt.close(fig)
    with zipfile.ZipFile(out/"trained_models.zip","w",compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for name,rawdata in model_data:z.writestr(name,rawdata)
    summary={"experiment":"EXP-CMB-004","status":"method completed, combined mechanism comparison pending",
      "method":args.method,"target_longside_pixels":goal,"foreground_threshold":THRESHOLD,
      "source_sha256":old_manifest["dataset_sha256"],
      "baseline_exp_cmb_001_oof_correct":2080,
      "handmade_writer_heldout":list(WRITERS),
      "official_train_n":76800,"official_val_n":19200,
      "per_fold_handmade_train_n":1920,"per_fold_validation_n":640,
      "trained_models_sha256":{k:hashlib.sha256(v).hexdigest() for k,v in model_data},
      "scikit_learn_version":sklearn.__version__,"numpy_version":np.__version__,
      "scipy_version":scipy.__version__,
      "no_test_used":True,"author_or_true_label_in_preprocessing":False,
      "method_explanatory_success_requires_not_accuracy_threshold":True,
      "training_seconds_total":sum(x["fit_seconds"] for x in fit_rows),
      "preprocess_seconds_total":preprocessing_seconds,"total_seconds":time.perf_counter()-start}
    (out/"run_manifest.json").write_text(json.dumps(summary,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print("COMPLETED",args.method,overall[0],flush=True)

if __name__=="__main__":
    main()


## pixel_transform

원본: [05_model_optimize/MLP/Combined/scripts/pixel_transform.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/pixel_transform.py)  
Git blob: `a56a88b602a98d0cf26dfa6ee36c31e2b9d08a7a`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

Byte-for-byte equivalent behavior to W02 experiment2_baseline/pixel_transform.py.



In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/pixel_transform.py
"""Byte-for-byte equivalent behavior to W02 experiment2_baseline/pixel_transform.py."""
from sklearn.base import BaseEstimator, TransformerMixin

class PixelScaler255(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X / 255.0

## exp_cmb004_compare

원본: [05_model_optimize/MLP/Combined/scripts/exp_cmb004_compare.py](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/scripts/exp_cmb004_compare.py)  
Git blob: `d5c097d4ca79560ad8a512c2aaa88e95b5bce791`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/05_model_optimize/MLP/Combined/logs)

Aggregate 4 newly trained geometry conditions into the preregistered comparison.

Never fit, tune, or select a model. This program presents paired evidence and
counterexamples, explicitly separating magnitude, error changes, official val
cost, and interaction from any final scientific judgement.

CLI 인자 (정적 추출): `--out`

In [ ]:
%%archive_source 05_model_optimize/MLP/Combined/scripts/exp_cmb004_compare.py
#!/usr/bin/env python3
"""Aggregate 4 newly trained geometry conditions into the preregistered comparison.

Never fit, tune, or select a model. This program presents paired evidence and
counterexamples, explicitly separating magnitude, error changes, official val
cost, and interaction from any final scientific judgement.
"""
import csv
import json
import sys
from collections import defaultdict
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support

ROOT=Path(__file__).resolve().parents[4]
CMB=ROOT/"05_model_optimize"/"MLP"/"Combined"
sys.path.insert(0,str(CMB/"scripts"))
import exp_cmb002_evidence as core

METHODS=("bbox_center","bbox_size","com_center","size_then_com")
LABELS=core.LABELS
WRITERS=core.WRITERS

def read(path):
    with path.open(encoding="utf-8-sig",newline="") as f:
        return list(csv.DictReader(f))

def write(path,rows):
    assert rows,path
    with path.open("w",encoding="utf-8-sig",newline="") as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]))
        w.writeheader();w.writerows(rows)

def num(x):
    return float(x)

def mean(rows,key):
    return sum(float(r[key]) for r in rows)/len(rows) if rows else float("nan")

def main():
    import argparse
    p=argparse.ArgumentParser()
    p.add_argument("--out",type=Path,required=True)
    args=p.parse_args()
    out=args.out
    results={}
    for m in METHODS:
        part=out/m
        manifest=json.loads((part/"run_manifest.json").read_text(encoding="utf-8"))
        assert manifest["method"]==m and manifest["no_test_used"]
        results[m]={
            "metrics":read(part/"metrics.csv"),
            "writer":read(part/"writer_metrics.csv"),
            "official":read(part/"official_val_metrics.csv"),
            "class":read(part/"class_metrics.csv"),
            "group":read(part/"group_metrics.csv"),
            "pairs":read(part/"pair_errors.csv"),
            "geo":read(part/"geometry_diagnostics.csv"),
            "focus":read(part/"focus_mechanism.csv"),
            "paired":read(part/"paired_predictions.csv"),
            "fit":read(part/"fit_metrics.csv"),
            "manifest":manifest}
        assert len(results[m]["paired"])==2560
        assert len(results[m]["focus"])==640
        assert len(results[m]["fit"])==4
    with (CMB/"results"/"EXP_CMB_001"/"metrics_summary.csv").open(encoding="utf-8-sig") as f:
        baseline=list(csv.DictReader(f))
    old_dev={int(r["held_out_writer"]):float(r["accuracy"]) for r in baseline
             if r["dataset"]=="official_dev" and r["model"]=="Combined"}
    assert set(old_dev)==set(WRITERS)
    _,predtable,oldtab,_,_=core.load_inputs()
    base_y=np.array([LABELS.index(r["true_label"]) for r in predtable])
    base_p=np.array([LABELS.index(r["combined_pred"]) for r in predtable])
    assert np.mean(base_y==base_p)==.8125

    summary=[{"method":"baseline","accuracy":.8125,"macro_f1":.8120430780652209,
      "correct":2080,"corrected":0,"induced":0,"delta_correct":0,"n":2560,
      "min_official_delta_pp":0,"max_official_delta_pp":0,
      "n_writers_improved":0,"n_writers_worsened":0,
      "mean_training_seconds_per_fold":"", "mean_n_iterations":""}]
    paired_all=[]
    writer_all=[];paired_by_method={}
    for method,d in results.items():
        r=d["metrics"][0]
        ov=[{"writer":int(z["writer"]),"delta_pp":100*(float(z["accuracy"])-old_dev[int(z["writer"])])}
             for z in d["official"]]
        wr=[{"writer":int(z["writer"]),"old":float(z["baseline_accuracy"]),
             "new":float(z["accuracy"])} for z in d["writer"]]
        summary.append({"method":method,
          "accuracy":r["accuracy"],"macro_f1":r["macro_f1"],"correct":r["correct"],
          "corrected":r["corrected"],"induced":r["induced"],
          "delta_correct":r["delta_correct"],"n":2560,
          "min_official_delta_pp":min(x["delta_pp"] for x in ov),
          "max_official_delta_pp":max(x["delta_pp"] for x in ov),
          "n_writers_improved":sum(x["new"]>x["old"] for x in wr),
          "n_writers_worsened":sum(x["new"]<x["old"] for x in wr),
          "mean_training_seconds_per_fold":mean(d["fit"],"fit_seconds"),
          "mean_n_iterations":mean(d["fit"],"n_iter")})
        writer_all.extend({"method":method,**z,"official_acc_delta_pp":
           next(x["delta_pp"] for x in ov if x["writer"]==int(z["writer"]))}
            for z in d["writer"])
        pp=d["paired"]
        for i,z in enumerate(pp):
            assert int(z["writer"])==int(predtable[i]["writer"])
            assert int(z["index_in_writer"])==int(predtable[i]["index_in_writer"])
            assert z["actual"]==predtable[i]["true_label"]
            assert z["baseline_pred"]==predtable[i]["combined_pred"]
        paired_by_method[method]=np.array([LABELS.index(x["prediction"]) for x in pp])
        paired_all.extend(pp)
    write(out/"method_comparison.csv",summary)
    write(out/"writer_comparison.csv",writer_all)

    # Aggregate all 4*4 directional pairs and 4 writers, preserving both errors and corrected/induced counts.
    errors=[]
    for method,d in results.items():
        for t,c in (("0","D"),("D","0"),("8","B"),("B","8")):
            z=[r for r in d["pairs"] if r["actual"]==t and r["rival"]==c]
            assert len(z)==4
            errors.append({"method":method,"actual":t,"rival":c,
                "baseline_errors":sum(int(r["baseline_errors"]) for r in z),
                "new_errors":sum(int(r["new_errors"]) for r in z),
                "delta_errors":sum(int(r["new_errors"])-int(r["baseline_errors"]) for r in z)})
    write(out/"main_error_pairs.csv",errors)

    group_rows=[]
    for method,d in results.items():
        for r in d["group"]:
            group_rows.append({"method":method,**r,
               "delta_accuracy_pp":100*(float(r["accuracy"])-float(r["baseline_accuracy"]))})
    write(out/"group_comparison.csv",group_rows)

    geometry=[]
    for method,d in results.items():
        for r in d["geo"]:
            geometry.append({"method":method,**r})
    write(out/"geometry_comparison.csv",geometry)

    # Scientific diagnostic: magnitude/outcome association within each
    # writer and actual-class; intentionally NOT using it to tune transforms.
    strata=[]
    for method,d in results.items():
        focus=d["focus"]
        for writer in WRITERS:
            for t in ("0","D","8","B"):
                subset=[r for r in focus if int(r["writer"])==writer and r["actual"]==t]
                assert len(subset)==40
                for state in ("corrected","new_error","both_correct","persistent_error"):
                    selected=[r for r in subset if r["transition"]==state]
                    if not selected:continue
                    abs_ratio=[abs(float(r["scale_factor"])-1) for r in selected]
                    abs_com=[abs(float(r["com_shift_px"])) for r in selected]
                    strata.append({"method":method,"writer":writer,"actual":t,
                        "transition":state,"count":len(selected),
                        "mean_abs_scale_factor_change":float(np.mean(abs_ratio)),
                        "mean_com_shift_px":float(np.mean(abs_com)),
                        "mean_bbox_long_change_px":mean(selected,"bbox_change"),
                        "mean_logit_margin_change":mean(selected,"margin_change"),
                        "mean_ink_mass_change_ratio":mean(selected,"mass_change_ratio")})
    write(out/"mechanism_by_stratum.csv",strata)

    # Changes are not caused solely by the image transform because weights are
    # re-fitted. This factorial contrast is descriptive of full train/infer pipelines.
    p0=base_p
    c=paired_by_method["bbox_size"]
    d=paired_by_method["com_center"]
    e=paired_by_method["size_then_com"]
    interaction=[]
    for kind,z in (("all",np.ones(2560,dtype=bool)),
                   ("digits",base_y<10),("letters",base_y>=10),
                   ("0",base_y==0),("D",base_y==13),
                   ("8",base_y==8),("B",base_y==11)):
        acc=lambda pred:float(np.mean(pred[z]==base_y[z]))
        interaction.append({"group":kind,"baseline":acc(p0),"size":acc(c),
                            "com":acc(d),"size_then_com":acc(e),
                            "descriptive_interaction_E_minus_C_minus_D_plus_A":
                            acc(e)-acc(c)-acc(d)+acc(p0)})
    write(out/"factorial_interaction.csv",interaction)

    fig,ax=plt.subplots(figsize=(9.5,4.8))
    xs=np.arange(len(summary))
    ax.plot(xs,[float(r["accuracy"])*100 for r in summary],marker="o",label="Hand-made OOF")
    ax.set(xticks=xs,xticklabels=[r["method"] for r in summary],
       ylabel="OOF Accuracy (%)",title="EXP-CMB-004 | same 4-Fold writer-held-out protocol")
    plt.setp(ax.get_xticklabels(),rotation=20,ha="right")
    ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(out/"accuracy_comparison.png",dpi=170);plt.close(fig)

    # Report transparency: no numerical gate, no automatic 'winner' label.
    lines=[
        "# EXP-CMB-004 — BBox Size vs CoM Center: 실측 비교 요약",
        "",
        "**통과 기준:** 사전등록 문서에 따라 특정 점수 이상이 아니라 어느 방법이 어떤 오류에 왜 효과적인지 **근거와 반례를 함께 설명**하는 것. 이 자동 보고서는 숫자만 제시하며 최종 연구 판정은 별도 실험일지에 작성한다.",
        "",
        "## 1. 전체 OOF (16 classes, 4 heldout writers)",
        "",
        "| Method | Accuracy | Macro F1 | Δ correct vs baseline | Corrected | Newly wrong | Writers improved | Official Val worst Δpp |",
        "|---|---:|---:|---:|---:|---:|---:|---:|",
    ]
    for r in summary:
        lines.append("| {} | {:.3%} | {:.4f} | {} | {} | {} | {} | {} |".format(
            r["method"],float(r["accuracy"]),float(r["macro_f1"]),
            r["delta_correct"],r["corrected"],r["induced"],r["n_writers_improved"],
            "—" if r["method"]=="baseline" else "{:+.3f}".format(float(r["min_official_delta_pp"]))))
    lines+=["","## 2. 주요 오답 방향별 Count","","| Method | 0→D | D→0 | 8→B | B→8 | 합계 |",
           "|---|---:|---:|---:|---:|---:|"]
    for method in METHODS:
        arr=[next(r for r in errors if r["method"]==method and r["actual"]==a and r["rival"]==b)
            for a,b in (("0","D"),("D","0"),("8","B"),("B","8"))]
        counts=[int(r["new_errors"]) for r in arr]
        lines.append("| {} | {} | {} | {} | {} | {} |".format(method,*counts,sum(counts)))
    lines+=["","## 3. Mechanism: 입력에서 실제로 변화한 특징",
           "",
           "| Method | Official train BBox long-side std before→after | Hand-made mean CoM offset before→after | Mean image mass change (Hand-made pooled by writers) |",
           "|---|---|---|---:|"]
    for method in METHODS:
        g=[r for r in geometry if r["method"]==method]
        off=next(x for x in g if x["sample"]=="official_train")
        hh=[x for x in g if x["sample"].startswith("writer_")]
        ink=sum(float(x["mean_abs_ink_mass_fraction_change"]) for x in hh)/4
        before=sum(float(x["mean_com_offset_before"]) for x in hh)/4
        after=sum(float(x["mean_com_offset_after"]) for x in hh)/4
        lines.append("| {} | {:.3f}→{:.3f} | {:.3f}→{:.3f}px | {:.4f} |".format(
           method,float(off["std_long_before"]),float(off["std_long_after"]),before,after,ink))
    lines+=["","## 4. Interaction (descriptive: 4 separate retrained models)",
        "",
        "| Group | E − Size − CoM + Baseline (accuracy percentage points) |",
        "|---|---:|"]
    for r in interaction:
        lines.append("| {} | {:+.3f} |".format(
           r["group"],100*r["descriptive_interaction_E_minus_C_minus_D_plus_A"]))
    lines+=["","### Interpretations that still need to be made",
        "",
        "- Size vs CoM must be interpreted using error-direction and writer-level transitions together with the actual size/center displacement and margin changes, not Accuracy rank alone.",
        "- Mean input mass change during scaling is NOT pure clipping loss: bilinear resizing changes area and intensity mass. Interpretation must separate scaling from cropping.",
        "- The combined-condition contrast is descriptive of retrained pipelines, NOT a clean causal estimate that holds model weights fixed.",
        "- Hypothesis evidence is developmental OOF on the same four previously examined writers; new-writer external verification is still missing.",
        "- Original Test and professor Test were never accessed.",
        "",
        "See: writer_comparison.csv, group_comparison.csv, main_error_pairs.csv, mechanism_by_stratum.csv, geometry_comparison.csv, factorial_interaction.csv and each method subfolder's actual PNG/CSV/models.",
        ""]
    (out/"SUMMARY.md").write_text("\n".join(lines),encoding="utf-8")
    (out/"AGGREGATE_MANIFEST.json").write_text(json.dumps({
        "experiment_id":"EXP-CMB-004",
        "methods":METHODS,"comparison_criterion":"explanatory evidence, no accuracy/F1 gate",
        "writer_heldout":[41,42,43,44],"official_test_used":False,
        "professor_final_test_used":False,"completed_method_count":len(results),
        "trained_model_count":len(results)*4,
        "baseline_retrained":False,
        "size_target":results["bbox_size"]["manifest"]["target_longside_pixels"],
        "method_model_member_hashes":{m:results[m]["manifest"]["trained_models_sha256"] for m in METHODS}
    },ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    print((out/"SUMMARY.md").read_text(encoding="utf-8"),flush=True)

if __name__=="__main__":
    main()


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")